In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from oracle.custom_datasets.BTS import BTS_val_parquet_path, BTS_test_parquet_path, BTS_train_parquet_path

In [ ]:
colors = {1: 'mediumaquamarine', 2: 'crimson', 3: 'goldenrod'}
filters = {1: 'g', 2: 'r', 3: 'i'}
markers = {1: 'o', 2: 's', 3: '^'}

In [ ]:
val_df = pd.read_parquet(BTS_val_parquet_path)
test_df = pd.read_parquet(BTS_test_parquet_path)


In [ ]:
df = pd.concat([val_df, test_df], ignore_index=True)

In [ ]:
def plot_ztf_sample(id, save=True):

    gs_kw = dict(width_ratios=[3, 1], height_ratios=[2, 1])
    fig, axd = plt.subplot_mosaic([['left', 'upper right'],
                                ['left', 'lower right']],
                                    gridspec_kw=gs_kw,
                                figsize=(9.5, 4), layout="constrained")
    for k in ['upper right', 'lower right']:
        axd[k].set_xticks([])
        axd[k].set_yticks([])

    sample = df[df['ZTFID'] == id]
    label = sample['bts_class'].to_numpy()[0]

    t = sample['jd'].to_numpy()[0]
    t = t - np.min(t)

    mag = sample['magpsf'].to_numpy()[0]
    mag_err = sample['sigmapsf'].to_numpy()[0]
    band = sample['fid'].to_numpy()[0]

    ra = sample['ra'].to_numpy()[0][0]
    dec = sample['dec'].to_numpy()[0][0]
    drb = sample['drb'].to_numpy()[0][0]
    sgscore = sample['sgscore1'].to_numpy()[0][0]
    scorr = sample['scorr'].to_numpy()[0][0]


    for b in np.unique(band):
        mask = band == b
        axd['left'].errorbar(t[mask], mag[mask], yerr=mag_err[mask], fmt=markers[b], label=filters[b], color=colors[b], mec='white', markersize=10)

    axd['left'].text(0.05, 0.1, f"{id} ({label})", transform=axd['left'].transAxes, fontsize='xx-large', verticalalignment='top')
    axd['left'].set_ylim(top=20.8)

    axd['left'].invert_yaxis()
    axd['left'].set_xlabel('Days since first detection', fontsize='xx-large')
    axd['left'].set_ylabel('mag', fontsize='xx-large')

    ps_data = sample['ps'].iloc[0]

    img = np.asarray(ps_data).reshape((3, 252, 252))
    img = np.permute_dims(img, (1,2,0))  # swap x and y for correct orientation

    axd['upper right'].imshow(img)


    text = f"RA: {ra:.3f}\nDec: {dec:.3f}\nSG Score: {sgscore:.2f}\nScorr: {scorr:.2f}"

    axd['lower right'].text(0, 0, text, transform=axd['lower right'].transAxes,
                        fontsize='xx-large', color='black', ha='left', va='bottom')   
    axd['lower right'].axis('off') 


    # make legend a separate figure
    fig_legend = plt.figure(figsize=(3, 1))
    ax_legend = fig_legend.add_subplot()
    for b in np.unique(band):
        mask = band == b
        ax_legend.plot([], [], markers[b], label=filters[b], color=colors[b])
    # make it 3 column
    ax_legend.legend(ncol=3)
    ax_legend.axis('off')


    if save:
        fig.savefig(f"{label}_{id}.pdf")
        fig_legend.savefig(f"legend.pdf")
    else:
        plt.show()

In [ ]:
plot_ztf_sample("ZTF21abaphri")

In [ ]:
plot_ztf_sample("ZTF20accbsxa")

In [ ]:
plot_ztf_sample("ZTF22aafrjnw")

In [ ]:
plot_ztf_sample("ZTF23aboebgh")

In [ ]:
plot_ztf_sample("ZTF20abgbulk")

In [ ]:
plot_ztf_sample("ZTF18abcwich")

In [ ]:
plot_ztf_sample("ZTF18abckxfb")

In [ ]:
# shuffle the df
df.sample(frac=1).reset_index(drop=True)